In [1]:
# ETAPA 1 - IMPORTAÇÃO E PRÉ-PROCESSAMENTO DA BASE
# -----------------------------------------------------------

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Importa os modelos
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Lê o arquivo CSV com os dados
df = pd.read_csv("Base_de_Dados.csv")

# Faz uma cópia para preservar o original
df_modelo = df.copy()

# Codifica variáveis categóricas ('Sim' → 1, 'Nao' → 0)
le = LabelEncoder()
for col in ['dor_no_peito', 'arterias_bloqueadas', 'doenca_coracao']:
    df_modelo[col] = le.fit_transform(df_modelo[col])

# Separa variáveis independentes (X) e dependente (y)
X = df_modelo.drop('doenca_coracao', axis=1)
y = df_modelo['doenca_coracao']

# Divide em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# ETAPA 2 - TREINAMENTO DOS MODELOS (BAGGING e RANDOM FOREST)
# -----------------------------------------------------------

# === BaggingClassifier ===
# Cria um modelo de Bagging, que treina várias árvores de decisão em subconjuntos aleatórios dos dados.
# Hiperparâmetros:
# - base_estimator=DecisionTreeClassifier(): usa árvores de decisão como modelo base
# - n_estimators=100: número de árvores no ensemble
# - random_state=42: garante reprodutibilidade dos resultados
bagging_model = BaggingClassifier(base_estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)

# Treina o modelo Bagging com os dados de treino
bagging_model.fit(X_train, y_train)

# Faz previsões com o conjunto de teste
y_pred_bagging = bagging_model.predict(X_test)

# Calcula a acurácia do modelo Bagging (proporção de acertos)
acc_bagging = accuracy_score(y_test, y_pred_bagging)

# === RandomForestClassifier ===
# Cria um modelo de Random Forest, que também é um ensemble de árvores de decisão com seleção aleatória de atributos.
# Hiperparâmetros:
# - n_estimators=100: número de árvores na floresta
# - random_state=42: controle de aleatoriedade para reprodutibilidade
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Treina o modelo Random Forest com os dados de treino
rf_model.fit(X_train, y_train)

# Faz previsões com o conjunto de teste
y_pred_rf = rf_model.predict(X_test)

# Calcula a acurácia do modelo Random Forest
acc_rf = accuracy_score(y_test, y_pred_rf)

In [4]:
# ETAPA 3 - AVALIAÇÃO DOS MODELOS
# -----------------------------------------------------------

def avaliar_modelo(nome_modelo, y_true, y_pred):
    print(f"\n==== {nome_modelo} ====")
    print("Matriz de Confusão:")
    print(confusion_matrix(y_true, y_pred))
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, y_pred, target_names=['Sem Doença', 'Com Doença']))

# Avaliação Bagging
avaliar_modelo("Bagging", y_test, y_pred_bagging)

# Avaliação Random Forest
avaliar_modelo("Random Forest", y_test, y_pred_rf)


==== Bagging ====
Matriz de Confusão:
[[ 6 12]
 [13  9]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Sem Doença       0.32      0.33      0.32        18
  Com Doença       0.43      0.41      0.42        22

    accuracy                           0.38        40
   macro avg       0.37      0.37      0.37        40
weighted avg       0.38      0.38      0.38        40


==== Random Forest ====
Matriz de Confusão:
[[ 4 14]
 [13  9]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Sem Doença       0.24      0.22      0.23        18
  Com Doença       0.39      0.41      0.40        22

    accuracy                           0.33        40
   macro avg       0.31      0.32      0.31        40
weighted avg       0.32      0.33      0.32        40



In [5]:
# ETAPA 4 - APLICAÇÃO DO ADABOOST E GRADIENT BOOSTING
# -----------------------------------------------------------

# === AdaBoostClassifier ===
# Cria um modelo de AdaBoost (Adaptive Boosting), que treina modelos fracos sequencialmente,
# ajustando os pesos dos exemplos mal classificados em cada etapa.
# Hiperparâmetros:
# - n_estimators=100: número de classificadores fracos (por padrão, árvores rasas)
# - random_state=42: define a semente para reprodutibilidade
adaboost_model = AdaBoostClassifier(n_estimators=100, random_state=42)

# Treina o modelo AdaBoost com os dados de treino
adaboost_model.fit(X_train, y_train)

# Realiza previsões com o conjunto de teste
y_pred_adaboost = adaboost_model.predict(X_test)

# Calcula a acurácia do modelo AdaBoost
acc_adaboost = accuracy_score(y_test, y_pred_adaboost)

# === GradientBoostingClassifier ===
# Cria um modelo de Gradient Boosting, que também combina modelos fracos sequencialmente,
# mas minimiza diretamente a função de perda por meio de gradientes.
# Hiperparâmetros:
# - n_estimators=100: número de árvores no ensemble
# - learning_rate=0.1: taxa de aprendizado, controla o peso de cada árvore adicionada
# - random_state=42: para resultados reproduzíveis
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Treina o modelo Gradient Boosting com os dados de treino
gb_model.fit(X_train, y_train)

# Realiza previsões com o conjunto de teste
y_pred_gb = gb_model.predict(X_test)

# Calcula a acurácia do modelo Gradient Boosting
acc_gb = accuracy_score(y_test, y_pred_gb)

# Avalia o desempenho do AdaBoost usando matriz de confusão e métricas de classificação
avaliar_modelo("AdaBoost", y_test, y_pred_adaboost)

# Avalia o desempenho do Gradient Boosting com as mesmas métricas
avaliar_modelo("Gradient Boosting", y_test, y_pred_gb)


==== AdaBoost ====
Matriz de Confusão:
[[ 8 10]
 [16  6]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Sem Doença       0.33      0.44      0.38        18
  Com Doença       0.38      0.27      0.32        22

    accuracy                           0.35        40
   macro avg       0.35      0.36      0.35        40
weighted avg       0.36      0.35      0.35        40


==== Gradient Boosting ====
Matriz de Confusão:
[[ 7 11]
 [14  8]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Sem Doença       0.33      0.39      0.36        18
  Com Doença       0.42      0.36      0.39        22

    accuracy                           0.38        40
   macro avg       0.38      0.38      0.37        40
weighted avg       0.38      0.38      0.38        40



In [7]:
# ETAPA 5 - APLICAÇÃO DO XGBOOST
# -----------------------------------------------------------
modelo_xgb = XGBClassifier(
    n_estimators=100,          # número de árvores
    learning_rate=0.1,         # taxa de aprendizado
    max_depth=3,               # profundidade máxima das árvores
    random_state=42,           # reprodutibilidade
)

# Treina o modelo XGBoost com os dados de treino
modelo_xgb.fit(X_train, y_train)

# Realiza previsões com o conjunto de teste
y_pred = modelo_xgb.predict(X_test)

# Calcula a acurácia do modelo XGBosst
acc_xgb = accuracy_score(y_test, y_pred)

# Avalia o desempenho do XGBoost usando matriz de confusão e métricas de classificação
avaliar_modelo("XGBoost", y_test, y_pred)


==== XGBoost ====
Matriz de Confusão:
[[ 7 11]
 [12 10]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Sem Doença       0.37      0.39      0.38        18
  Com Doença       0.48      0.45      0.47        22

    accuracy                           0.42        40
   macro avg       0.42      0.42      0.42        40
weighted avg       0.43      0.42      0.43        40



In [8]:
# ETAPA 6 - COMPARAÇÃO FINAL ENTRE TODOS OS MÉTODOS
# -----------------------------------------------------------

# Tabela com as acurácias de todos os modelos
resultado_comparativo_final = pd.DataFrame({
    'Modelo': ['Bagging', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Acurácia': [acc_bagging, acc_rf, acc_adaboost, acc_gb, acc_xgb]
})

# Exibe a tabela resumo
print("\n=== Comparação Final de Acurácias ===")
print(resultado_comparativo_final)


=== Comparação Final de Acurácias ===
              Modelo  Acurácia
0            Bagging     0.375
1      Random Forest     0.325
2           AdaBoost     0.350
3  Gradient Boosting     0.375
4            XGBoost     0.425
